# Building a sample dataset for testing derivative products across different Landsat sensors

This notebook is designed to create a sample set for testing derivative products across the continent, where the testing needs to compare data derived from multiple landsat sensors. It is designed to do the following:

- Import a geojson containing the footprints of the 'golden tiles' that have been previously used for validating GeoMAD products. These tiles are distributed across Australia and across a range of environment types.
- Load a datacube dataset for the selected sensors (in this case, Landsat 7, and Landsat 8/9), using one or more of the golden tiles as the geometry for the datacube query
- Compare the resulting datasets and find the timesteps that overlap ( +/- a timeframe set by the user, e.g. 48 hours)
- If the resulting filtered dataset is very large, create a subset based on randomly selecting smaller regions or pixels
- Export the resulting geojson so it can be used as an input to test workflows

In [1]:
!pip uninstall dea-tools -y

In [2]:
import os
import datacube
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import pprint
from datetime import timedelta
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS, Geometry, GeoBox
from datacube.utils import masking
from pathlib import Path
from shapely.geometry import box

import pystac_client
import planetary_computer

import odc.geo.xr
from odc.geo.xr import assign_crs
from odc.io.cgroups import get_cpu_quota
from odc.geo.geom import BoundingBox

import sys

sys.path.insert(1, "../../../Tools")
from dea_tools.datahandling import load_ard
from dea_tools.classification import collect_training_data, HiddenPrints
from dea_tools.dask import create_local_dask_cluster
from dea_tools.plotting import rgb, display_map
from dea_tools.spatial import xr_vectorize, xr_rasterize
from dea_tools.bandindices import calculate_indices

import warnings

warnings.filterwarnings("ignore")


In [3]:
client = create_local_dask_cluster(return_client=True)


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/36221/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/36221/status,Workers: 1
Total threads: 15,Total memory: 114.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43045,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/36221/status,Total threads: 15
Started: Just now,Total memory: 114.00 GiB
Comm: tcp://127.0.0.1:35287,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/39555/status,Memory: 114.00 GiB
Nanny: tcp://127.0.0.1:34067,


In [4]:
# dc = datacube.Datacube()


In [5]:
# funtion from Chad to sample ABARES landuse data

def random_sampling(
    da, n, sampling="stratified_random", manual_class_ratios=None, out_fname=None
):
    """
    Creates randomly sampled points for post-classification
    accuracy assessment.

    Params:
    -------
    da: xarray.DataArray
        A classified 2-dimensional xarray.DataArray
    n: int
        Total number of points to sample. Ignored if providing
        a dictionary of {class:numofpoints} to 'manual_class_ratios'
    sampling: str
        'stratified_random' = Create points that are randomly
        distributed within each class, where each class has a
        number of points proportional to its relative area.
        'equal_stratified_random' = Create points that are randomly
        distributed within each class, where each class has the
        same number of points.
        'random' = Create points that are randomly distributed
        throughout the image.
        'manual' = user definined, each class is allocated a
        specified number of points, supply a manual_class_ratio
        dictionary mapping number of points to each class
    manual_class_ratios: dict
        If setting sampling to 'manual', the provide a dictionary
        of type {'class': numofpoints} mapping the number of points
        to generate for each class.
    out_fname: str
        If providing a filepath name, e.g 'sample_points.shp', the
        function will export a shapefile/geojson of the sampling
        points to file.

    Output
    ------
    GeoPandas.Dataframe

    """

    if sampling not in [
        "stratified_random",
        "equal_stratified_random",
        "random",
        "manual",
    ]:
        raise ValueError(
            "Sampling strategy must be one of 'stratified_random', "
            + "'equal_stratified_random', 'random', or 'manual'"
        )

    # open the dataset as a pandas dataframe
    da = da.squeeze()
    df = da.to_dataframe() #name="class"
    df = df.dropna(how="any")

    # list to store points
    samples = []

    if sampling == "stratified_random":
        # determine class ratios in image
        class_ratio = pd.DataFrame(
            {
                "proportion": df["class"].value_counts(normalize=True),
                "class": df["class"].value_counts(normalize=True).keys(),
            }
        )

        for _class in class_ratio["class"]:
            # use relative proportions of classes to sample df
            no_of_points = (
                n * class_ratio[class_ratio["class"] == _class]["proportion"].values[0]
            )
            # random sample each class
            print(
                "Class "
                + str(_class)
                + ": sampling at "
                + str(round(no_of_points))
                + " coordinates"
            )
            sample_loc = df[df["class"] == _class].sample(n=int(round(no_of_points)))
            samples.append(sample_loc)

    if sampling == "equal_stratified_random":
        classes = np.unique(df["class"])

        for _class in classes:
            # use relative proportions of classes to sample df
            no_of_points = n / len(classes)
            # random sample each classes
            try:
                sample_loc = df[df["class"] == _class].sample(
                    n=int(round(no_of_points))
                )
                print(
                    "Class "
                    + str(_class)
                    + ": sampling at "
                    + str(round(no_of_points))
                    + " coordinates"
                )
                samples.append(sample_loc)

            except ValueError:
                print(
                    "Requested more sample points than population of pixels for class "
                    + str(_class)
                    + ", skipping"
                )
                pass

    if sampling == "random":
        no_of_points = n
        # random sample entire df
        print(
            "Randomly sampling dataAraay at "
            + str(round(no_of_points))
            + " coordinates"
        )
        sample_loc = df.dropna().sample(n=int(round(no_of_points)))
        samples.append(sample_loc)

    if sampling == "manual":
        if isinstance(manual_class_ratios, dict):
            # check classes in dict match classes in data
            classes = np.unique(df["class"])
            dict_classes = list(manual_class_ratios.keys())

            if set(dict_classes).issubset([str(i) for i in classes]):
                # mask for just those classes in the provided dictionary
                mask = np.isin(classes, np.array(dict_classes).astype(type(classes[0])))
                classes = classes[mask]
                # run sampling
                for _class in classes:
                    no_of_points = manual_class_ratios.get(str(_class))
                    # random sample each class
                    try:
                        sample_loc = df[df["class"] == _class].sample(
                            n=int(round(no_of_points))
                        )
                        print(
                            "Class "
                            + str(_class)
                            + ": sampled at "
                            + str(round(no_of_points))
                            + " coordinates"
                        )
                        samples.append(sample_loc)

                    except ValueError:
                        print(
                            "Requested more sample points than population of pixels for class "
                            + str(_class)
                            + ", skipping"
                        )
                        pass

            else:
                raise ValueError(
                    "Some or all of the classes in 'manual_class_ratio' dictionary do not"
                    + " match the classes in the supplied dataArray. "
                    + "DataArray classes: "
                    + str(classes)
                    + ", Supplied dict classes: "
                    + str(list(manual_class_ratios.keys()))
                )

        else:
            raise ValueError(
                "Must supply a dictionary mapping {'class': numofpoints} if sampling"
                + " is set to 'manual'"
            )

    # join back into single datafame
    all_samples = pd.concat([samples[i] for i in range(0, len(samples))])

    # get pd.mulitindex coords as list
    y = [i[0] for i in list(all_samples.index)]
    x = [i[1] for i in list(all_samples.index)]

    # create geopandas dataframe
    gdf = gpd.GeoDataFrame(
        all_samples, crs=f"EPSG:{da.odc.crs.epsg}", geometry=gpd.points_from_xy(x, y)
    ).reset_index()

    gdf = gdf.drop(["x", "y"], axis=1)

    if out_fname is not None:
        gdf.to_file(out_fname)

    return gdf


In [6]:
def get_abares_classes(abares_year='2020', sample_size=1000):
    #There are 2 years of ABARES CLUM data in the datacube, select one.
    if abares_year =='2020':
        product = "abares_clum_2020"
    elif abares_year == '2023':
        product = "abares_clum_2023"
    else:
        print("Please enter valid ABARES CLUM year: 2020 or 2023.")

    query_abares = {
        "resolution": resolution,
        "output_crs": output_crs,
        "group_by": "solar_day",
        "product": product,
        "resolution": (-30, 30),
        "dask_chunks": {"time":1, "x":2048, "y": 2048}
    }

    #modify the abares query to select a golden tile. This will be removed at a later date.
    #query_abares = select_tile(test_tiles_gdf, region_codes, query_abares)

    ds_clum = dc.load(**query_abares)

    ds_clum_classes = (
        ds_clum // 100
    ) * 100  # convert all the classes to only have the parent 6 classes for now.

    ds_output = ds_clum_classes.alum_class
    ds_output = ds_output.squeeze().drop_vars("time")

    return ds_output

In [7]:
def feature_layers(query):
    dc = datacube.Datacube()
    ds = dc.load(product='ga_ls8cls9c_gm_cyear_3',
                measurements=[
                "nbart_blue",
                "nbart_green",
                "nbart_red",
                "nbart_nir"],
                **query)
                

    # bandnames_dict = {
    #             "blue": "nbart_blue",
    #             "green": "nbart_green",
    #             "red": "nbart_red",
    #             "nir": "nbart_nir"
    #         }
    # ds.rename(bandnames_dict)

    
    ds = calculate_indices(
        ds, index=["NDVI"], drop=False, collection="ga_ls_3"
    ) #, "TCW_DEA", "TCB_DEA", "TCG_DEA", "TCW_ls8", "TCB_ls8", "TCG_ls8"

    return ds


### Analysis parameters


In [8]:
output_crs = "EPSG:3577"

time = ('2020-01', '2020-12')

au_boundary_dir = "australia_100km_buffer.geojson"

In [9]:
au_boundary_gpd = gpd.read_file(au_boundary_dir)

xmin, ymin, xmax, ymax = au_boundary_gpd.total_bounds

x = (122.10, 122.48)
y = (-17.91, -18.28)

# x = (111, 154)
# y = (-8, -45)

bbox = BoundingBox.from_xy(x, y)
time_range = "/".join(time)


print(f"xmin: {xmin}, xmax:{xmax}, ymin: {ymin}, ymax:{ymax}")
print(au_boundary_gpd.total_bounds)

xmin: 111.9154159687281, xmax:154.67347079958884, ymin: -44.79649874840166, ymax:-8.20633892690521
[111.91541597 -44.79649875 154.6734708   -8.20633893]


In [10]:
# Try using ESA worldvoer instead of ABARES

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


In [11]:

# Search for STAC items from "esa-worldcover" product
search = catalog.search(
    collections=["esa-worldcover"],
    bbox=bbox,
    datetime = time_range
)

# Check how many items were returned
items = list(search.get_items())
#items

In [12]:
ds_worldcover = odc.stac.load(
    items,
    bbox=bbox,
    bands=['map'],
    crs = 'EPSG:3577',
    resolution=60
)



In [13]:
ds_worldcover = ds_worldcover.squeeze().drop_vars('time')
ds_worldcover = ds_worldcover.rename({'map': 'class'})

In [14]:
fname_outpath_worldcover_samples = f"worldcover_stratified_sample_points_2020.gpkg"

worldcover_sample = random_sampling(
    ds_worldcover, 
    100, 
    sampling="equal_stratified_random", 
    out_fname=fname_outpath_worldcover_samples
)

Class 10: sampling at 11 coordinates
Class 20: sampling at 11 coordinates
Class 30: sampling at 11 coordinates
Class 40: sampling at 11 coordinates
Class 50: sampling at 11 coordinates
Class 60: sampling at 11 coordinates
Class 80: sampling at 11 coordinates
Class 90: sampling at 11 coordinates
Class 95: sampling at 11 coordinates


### ALUM primary classes:

TODO: refine classes to use in TC analysis so we can compare tree vs shrubs etc.

- 1: conservation and natural environments
- 2: production from relatively natural environments
- 3: production from dryland agriculture and plantations
- 4: production from irrigated agriculture and plantations
- 5: intensive uses
- 6: water (note wetlands are 651 for conservation)

### Collect stratified random samples from ABARES CLUM and save out as geopackage

In [15]:
# fname_outpath_abares_samples = f"abares_stratified_sample_points_2020.gkpg"

# abares_classes = get_abares_classes(abares_year='2020', sample_size=1000)

In [16]:
# abares_sample = random_sampling(
#     abares_classes, 
#     2000, 
#     sampling="equal_stratified_random", 
#     out_fname=fname_outpath_abares_samples
# )

## Set up query and function for collecting data from datacube, stop dask client

- the `collect_sample_data` function doesn't play nicely with a local dask cluster, so the local dask client needs to be shut down before running that cell.

In [17]:
client.shutdown()

In [18]:
ncpus = round(get_cpu_quota())


In [19]:
# read the geopackage file back in to then collect sample data from datacube for TC's

samples_gpd = gpd.read_file(fname_outpath_worldcover_samples)

query = {
    "time": ('2020'),
    "resolution": (-900,900),
    "output_crs": output_crs
}

In [24]:
samples_gpd.explore()

In [20]:
class_list = list(samples_gpd['class'].unique())

In [23]:
# column_names_all =[]
# model_input_all = []

# for class_id in class_list:
#     samples = samples_gpd[samples_gpd['class']==class_id]
#     print(class_id)

    # with HiddenPrints():
column_names, model_input = collect_training_data(
    gdf=samples,
    dc_query=query,
    ncpus=1,
    #return_coords=True,
    field="class",
    #zonal_stats="mean",  # not actually going to use this but it complains if set to False.
    feature_func=feature_layers,
)
# column_names_all = column_names
# model_input_all.append(model_input)
        

IndexError: single positional indexer is out-of-bounds

In [ ]:
output_file = f"worldcover_samples_tassel_caps_2020.txt"

np.savetxt(output_file, model_input, header=" ".join(column_names), fmt="%4f")